# Broader original-source corpus intake

Use a **CPU runtime**. The default `core` profile adds **39.011 GB (36.332 GiB)**
of original releases: DroneNoise, Berne, Polish UAV/speech, NEAPTIDE, both TUT2017
releases, UrbanSound8K and DNC. Keep the completed AuDroK intake and its receipts.
The files include repeated microphones, calibration and processed measurements;
downloaded bytes do not establish independent hours, labels or field reliability.

Check available Drive space before running. `STORAGE_BUDGET_GIB` is your declared
allocation for this selection; it does not measure the account quota. The separate
folder below preserves earlier code pins and completed sources. Use one active
runtime for this folder. No packages, GPU allocation or model training are needed.


In [ ]:
PROJECT_DRIVE_PATH = '/content/drive/MyDrive/drone-detection-v2/corpus-intake-v3'
INTAKE_PROFILE = 'core'  # Alternatives: 'field', 'background', 'metadata'.
STORAGE_BUDGET_GIB = 60.0  # Set from the available Drive allocation you have checked.
REPO_URL = 'https://github.com/StoneDerek/drone-detection.git'
FIRST_REF = 'main'


In [ ]:
import os
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
MOUNT = Path('/content/drive')
if not os.path.ismount(MOUNT):
    raise RuntimeError('Drive is not mounted; refusing an ephemeral intake folder')
PROJECT = Path(PROJECT_DRIVE_PATH).resolve()
if PROJECT == MOUNT.resolve() or not PROJECT.is_relative_to(MOUNT.resolve()):
    raise ValueError('Choose a project folder inside mounted Drive')
PROJECT.mkdir(parents=True, exist_ok=True)


In [ ]:
"""Standard-library-only bootstrap, embedded verbatim in the Colab notebook."""

import hashlib
import json
import os
import re
import subprocess
import uuid
from pathlib import Path


def git_output(checkout, *arguments):
    return subprocess.check_output(['git', '-C', str(checkout), *arguments], text=True).strip()


def file_hash(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def pin_checkout(project, checkout, repo_url, first_ref='main'):
    """Resolve once, then reuse that exact revision after runtime replacement.

    One active runtime per project. Refuse dirty or unrelated local checkouts.
    The saved lock hash is also checked; a pin is never silently refreshed.
    """
    project, checkout = Path(project), Path(checkout)
    project.mkdir(parents=True, exist_ok=True)
    pin_path = project / 'bootstrap.json'
    pin = json.loads(pin_path.read_text()) if pin_path.exists() else None
    if pin is not None and (pin.get('schema_version') != 1
                            or pin.get('repo_url') != repo_url
                            or not re.fullmatch(r'[0-9a-f]{40}', pin.get('revision', ''))
                            or not re.fullmatch(r'[0-9a-f]{64}', pin.get('runtime_lock_sha256', ''))):
        raise ValueError('Invalid/different project pin; choose a new project folder deliberately')
    if checkout.exists() and any(checkout.iterdir()):
        if not (checkout / '.git').exists():
            raise ValueError('Checkout folder contains unrelated files')
        if git_output(checkout, 'remote', 'get-url', 'origin') != repo_url:
            raise ValueError('Checkout has a different repository origin')
        if git_output(checkout, 'status', '--porcelain'):
            raise ValueError('Local checkout changed; preserve those changes before continuing')
    else:
        checkout.mkdir(parents=True, exist_ok=True)
        subprocess.run(['git', 'init', '--quiet', str(checkout)], check=True)
        subprocess.run(['git', '-C', str(checkout), 'remote', 'add', 'origin', repo_url], check=True)
    target = pin['revision'] if pin else first_ref
    subprocess.run(['git', '-C', str(checkout), 'fetch', '--quiet', '--depth', '1', 'origin', target], check=True)
    revision = git_output(checkout, 'rev-parse', 'FETCH_HEAD')
    if pin and revision != pin['revision']:
        raise ValueError('Fetched revision differs from saved project pin')
    subprocess.run(['git', '-C', str(checkout), 'checkout', '--quiet', '--detach', revision], check=True)
    candidate = {'schema_version': 1, 'repo_url': repo_url, 'revision': revision,
                 'runtime_lock_sha256': file_hash(checkout / 'notebooks/runtime/uv.lock')}
    if pin is not None and pin != candidate:
        raise ValueError('Pinned environment lock changed')
    if pin is None:
        payload = json.dumps(candidate, indent=2, sort_keys=True).encode()
        temporary = pin_path.with_name('bootstrap.partial-' + uuid.uuid4().hex)
        temporary.write_bytes(payload)
        if pin_path.exists() and json.loads(pin_path.read_text()) != candidate:
            raise ValueError('Another bootstrap wrote a different pin; use one active runtime')
        os.replace(temporary, pin_path)
        if pin_path.read_bytes() != payload:
            raise ValueError('Project pin read-back failed')
    return candidate


REPO = Path('/content/drone-corpus-intake-v3')
PIN = pin_checkout(PROJECT, REPO, REPO_URL, FIRST_REF)
print(json.dumps(PIN, indent=2))


In [ ]:
import subprocess
import sys

COMMAND = [sys.executable, str(REPO / 'scripts/fetch_corpus_intake.py'),
           '--manifest', str(REPO / 'data/corpus-intake-v3.json'),
           '--profile', INTAKE_PROFILE,
           '--output', str(PROJECT / 'source-archives')]
subprocess.run(COMMAND + ['--plan-only'], cwd=REPO, check=True)


Each completed file must match its pinned publisher checksum. The stored
file is read back to check that checksum and record SHA-256 before completion is
published. On a replacement runtime, completed files are rehashed and reused.
Interrupted files retain an **unverified partial** and an identity journal.
Rerunning resumes its exact byte offset where the server supports Range; a server
that ignores Range restarts that file. A changed source or corrupt prefix cannot
pass the final checksum. Preserve failed files for review rather than deleting
the intake folder.

This freezes acquisition state only. Keep native originals and per-source terms.
Do not extract the whole selection onto the notebook SSD. Recording/channel
parentage and interval labels will be prepared in a separate bounded cache step.


In [ ]:
subprocess.run(COMMAND + ['--budget-gib', str(STORAGE_BUDGET_GIB)],
               cwd=REPO, check=True)
print('Original distributions:', PROJECT / 'source-archives')


Flush before ending a session. After replacing the runtime, rerun this
notebook with the same folder to verify completed-file persistence and partial
recovery. Mounted read-back is not independent proof of cloud sync; unexpected
termination can lose bytes not yet synchronized to Drive.

Source/label inspection and a grouped manifest follow acquisition. The 100-hour
target was an unvalidated estimate; training quantity will be assessed using
grouped learning curves and separate held-out evaluation exposure.


In [ ]:
drive.flush_and_unmount()
print('Drive flushed and unmounted. Verify persistence on the replacement runtime.')
